# GPU Notebook - Physics Units Interpretability

Companion to `run_gpu.ipynb` (capitals) and `run_gpu_math.ipynb` (addition/carry),
targeting the project's **science/numeracy extension**: the **physics-units** behaviour
(mapping a physical quantity to its SI unit).

**Key difference:** the SAEs were trained on *pooled* activations from all three
behaviours (capitals + addition + units), so they already cover units. This notebook
**reuses the existing SAE checkpoints** and therefore **skips activation capture and
SAE training entirely**.

### The behaviour
Each prompt asks for the SI unit of a physical quantity, e.g.

> Fact: The official SI unit for the **force** of a moving engine thrust is named "

The model should answer `newtons`. The dataset (`data/generate_datasets.py`,
`generate_units()`) pairs each quantity with a physically-implausible distractor unit
(force -> `volts`, energy -> `ohms`, mass -> `seconds`, ...), so we can build a clean
**contrast graph** exactly as the carry notebook did with `4` vs `3`.

### The clean intervention (minimal pair)
The strongest experiment mirrors the addition notebook's minimal-pair carry patch. We
use two prompts that are **identical except the quantity word**:

| Prompt | quantity | expected unit |
|--------|----------|---------------|
| `... unit for the **force** of a moving engine thrust ...`  | force  | **`newtons`** |
| `... unit for the **energy** of a moving engine thrust ...` | energy | **`joules`**  |

If `force` and `energy` tokenize to the same length, the prompts line up
position-for-position, so we can **activation-patch** the energy run into the force run
and watch `newtons` fall / `joules` rise. This isolates the *quantity->unit* mechanism
the same way the carry pair isolated carry logic.

**Runtime settings:** GPU (any tier), High-RAM recommended.

## Step 1 - Mount Drive & fetch project code

GitHub is the primary code source; Drive `project.zip` remains the backup. SAE checkpoints still restore from Drive.


In [ ]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary source for code. Drive project.zip is kept as a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"
use_github_code = True

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
if use_github_code:
    try:
        clone_dir = repo_dir
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        else:
            if os.path.exists(clone_dir) and os.listdir(clone_dir):
                clone_dir = "/content/test_run_github"
            if os.path.isdir(os.path.join(clone_dir, ".git")):
                run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
            elif os.path.exists(clone_dir) and os.listdir(clone_dir):
                raise RuntimeError(f"{clone_dir} exists but is not a git repo")
            else:
                run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
        os.chdir(clone_dir)
        github_ok = True
        print(f"Using GitHub checkout: {os.getcwd()}")
    except Exception as exc:
        print("GitHub checkout failed; falling back to Drive project.zip.")
        print(repr(exc))

if not github_ok:
    uploaded_directly_to_colab = False
    zip_path = "/content/project.zip" if uploaded_directly_to_colab else "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"Could not find {zip_path}. Upload project.zip to Drive/Colab, or fix GitHub access."
        )
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


## Step 2 - Install dependencies

In [ ]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

## Step 3 - Generate the units dataset
Only the physics-units CSV is needed here.

In [ ]:
# Step 3: Generate Datasets (units CSV is what we need)
!python data/generate_datasets.py
print("\nDataset files:")
!ls -lh data/units_data.csv

## Step 4 - Restore SAE checkpoints from Drive (no training)

The SAEs are behaviour-agnostic (trained on pooled activations), so we reuse the
checkpoints produced by `run_gpu.ipynb`. **If they are missing**, run `run_gpu.ipynb`
Step 5 first.

In [ ]:
# Step 4: Restore SAE checkpoints from Drive
import os, shutil, glob

drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
local_sae = '/content/mechanistic_data/sae_checkpoints'
os.makedirs(local_sae, exist_ok=True)

if os.path.isdir(drive_sae) and glob.glob(f'{drive_sae}/sae_layer*.pt'):
    for f in os.listdir(drive_sae):
        shutil.copy2(f'{drive_sae}/{f}', local_sae)
    print("Restored SAE checkpoints from Drive:")
    !ls -lh {local_sae}
else:
    print("No SAE checkpoints found in Drive at:", drive_sae)
    print("Run run_gpu.ipynb Step 5 (training) first.")

In [ ]:
# Refresh source code from GitHub; Drive Python files are backup only.
# This replaces the old Drive-first script overwrite flow.
import glob
import os
import shutil
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_root = os.getcwd()
try:
    if not os.path.isdir(os.path.join(repo_root, ".git")):
        raise RuntimeError(f"Current directory is not a git checkout: {repo_root}")
    print("Refreshing code from GitHub...")
    subprocess.run(["git", "pull", "--ff-only"], check=True)
    print("GitHub refresh complete.")
except Exception as exc:
    print("GitHub refresh failed; falling back to Drive Python-file backup.")
    print(repr(exc))
    os.makedirs("/content/src", exist_ok=True)
    for src in glob.glob("/content/drive/MyDrive/mphil-project/*.py"):
        shutil.copy2(src, "/content/src")
        print("Copied backup", os.path.basename(src))

# Legacy Drive backup commands kept for reference, but GitHub is preferred:
# !rm /content/src/attribution_graph.py
# !rm /content/src/intervention.py
# !cp /content/drive/MyDrive/mphil-project/*.py /content/src

!python src/intervention.py --help | head -30


---
## Baseline - does the model know the units?

Quick check that the model actually produces the right SI unit for the units behaviour
before we interpret it. This runs the standard baseline over the units prompt set.

In [ ]:
# Baseline over the units behaviour only (prefix-guided so plural/singular unit forms count)
!python src/baseline.py --config configs/baseline_config.yaml --prefix 3
print("\nUnits baseline summary:")
!python - <<'PY'
import json
d=json.load(open('outputs/baselines/units_baselines.json'))
n=len(d); top1=sum(1 for r in d if r['expected_rank']==0); intk=sum(1 for r in d if r['expected_rank'] is not None)
print(f"units: {top1}/{n} top-1 correct, {intk}/{n} in top-k")
PY

---
## Attribution graphs - units prompts (contrast)

As in the carry notebook, we build a **contrast graph**: instead of attributing the raw
`newtons` logit, we attribute `logit(newtons) - logit(joules)` for a force prompt. This
targets *what makes the model prefer the correct unit over a physically-related wrong
unit*, which is the interpretable quantity for the units behaviour.

We build graphs for two quantities so the intervention cells have targets:
- **force** -> `newtons` (contrast `joules`)
- **energy** -> `joules` (contrast `newtons`)

Each graph prints a ready-to-paste `--features` string. We use `--layers 4 8 12 16 20 24 28`
(layer-32 SAE is degenerate, same as the other notebooks).

In [ ]:
# Graph 1: force -> newtons (contrast joules). Key units contrast graph.
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target "newtons" \
  --contrast-target "joules" \
  --layers 4 8 12 16 20 24 28 \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_force_graph.json \
  --output-html outputs/units_force_graph.html \
  --output-mermaid outputs/units_force_graph.md

In [ ]:
# Graph 2: energy -> joules (contrast newtons). Reverse contrast for the minimal-pair swap.
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --target "joules" \
  --contrast-target "newtons" \
  --layers 4 8 12 16 20 24 28 \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_energy_graph.json \
  --output-html outputs/units_energy_graph.html \
  --output-mermaid outputs/units_energy_graph.md

In [ ]:
# Persist attribution graphs immediately (crash-safe)
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in glob.glob('/content/outputs/units_*graph*'):
    shutil.copy2(f, drive_out)
    print("Copied", os.path.basename(f))

---
## Diagnostic 1 - Component knockouts

Same battery as the carry notebook. MLP knockout tests the SAE-feature path; attention
and block knockouts test whether the decisive unit signal lives outside the final-token
MLP. We run both final-token and all-position variants, since the carry notebook found
all-position MLP knockout was far more informative than final-token-only.

In [ ]:
# Final-token MLP knockout (force -> newtons), tracking newtons / joules / volts
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component mlp \
  --output outputs/units_knockout_mlp_force.json

In [ ]:
# Token map: choose positions for the all-position / minimal-pair cells below
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --print-tokens

In [ ]:
# All-position MLP knockout with per-layer scan (the informative variant from the carry work)
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component mlp \
  --layer-scan \
  --positions all \
  --output outputs/units_knockout_mlp_allpos_force.json

In [ ]:
# All-position attention knockout (component-level diagnostic)
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --full-knockout \
  --knockout-component attn \
  --positions all \
  --output outputs/units_knockout_attn_allpos_force.json

## Diagnostic 2 - Progressive SAE contrast-feature ablation

Ablate graph-positive features from the `newtons - joules` force contrast graph. Watch
the gap between `newtons` and `joules`, not just the absolute `newtons` logit.

In [ ]:
# Progressive scan of the force contrast graph
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/units_force_graph.json \
  --graph-feature-sign positive \
  --scan \
  --output outputs/units_scan_force.json

In [ ]:
# Inhibit ALL force-graph positive features at once
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/units_force_graph.json \
  --graph-feature-sign positive \
  --output outputs/units_inhibit_force.json

---
## Main result - minimal-pair quantity swap (force <-> energy)

The cleanest units experiment, mirroring the carry minimal-pair patch. The two prompts
differ **only in the quantity word** (`force` vs `energy`), same template and same object
(`engine thrust`):

- **force** prompt -> `newtons`
- **energy** prompt -> `joules`

Both prompts tokenize to **17 tokens and differ at exactly one position** (position 8:
`' force'` vs `' energy'`), so they line up position-for-position. We patch the energy
activations into the force run across all positions and watch `newtons` fall / `joules`
rise. A clean quantity->unit mechanism should transfer the unit identity. We add
`--layer-scan` to localize which layer carries the effect, and run both directions as a
symmetry check.

> **Multi-token targets.** The unit words are multiple tokens (`newtons` = `new`+`tons`,
> `joules` = `j`+`ou`+`les`), but `intervention.py` tracks each `--target-token` by its
> **first** token id, and those first tokens are distinct (`new` / `j` / `vol`), so the
> logit/prob deltas are well-defined. We report the first-token logit, which is what the
> model commits to at the answer position.
>
> **Robustness.** The position-aware swap enforces equal-length prompts and prints a clear
> error if a chosen pair tokenizes differently. To swap a different quantity pair whose
> lengths differ, fall back to `--positions last` (patches only the final token, still a
> valid transfer probe).

In [ ]:
# Token maps for both prompts: confirm equal length before the all-position swap
!python src/intervention.py --mode inhibit --print-tokens \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "'
print("=" * 60)
!python src/intervention.py --mode inhibit --print-tokens \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' 

In [ ]:
# MAIN: patch ENERGY latent code into the FORCE run, all positions, per-layer scan.
# Clean quantity->unit signal = newtons falls and joules rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_swap_minpair_energy_to_force.json

In [ ]:
# SYMMETRY CHECK (reverse): patch FORCE into the ENERGY run.
# Clean effect = joules falls and newtons rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_swap_minpair_force_to_energy.json

In [ ]:
# SPARSE version: swap only the force-graph positive features (not the full latent code).
# If the full swap moves newtons/joules but this sparse one does not, the unit signal is
# real but not captured by the SAE's top features (same limitation story as the carry work).
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --graph-json outputs/units_force_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_swap_minpair_sparse_energy_to_force.json

---
## Copy all outputs to Drive

In [ ]:
# Copy all units outputs to Google Drive for persistence
import os, shutil, glob
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/outputs/units_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')

---
## Notes on interpreting these results

- **Contrast graph, not raw logit.** As with carry, the interpretable question is
  *newtons vs a physically-related wrong unit (joules)*, so graphs use
  `--target newtons --contrast-target joules`.
- **Minimal pair is the clean intervention.** `force` vs `energy` on an identical
  template isolates the quantity->unit mapping the way `58+83` vs `54+83` isolated carry.
  The full-latent swap is the strong probe; the sparse graph-feature swap tests whether
  the SAE's named features actually implement it.
- **All-position beats final-token.** The carry notebook found final-token-only MLP
  knockout uninformative but all-position knockout strong; we run both here for units.
- **Watch the gap.** Report `logit(newtons) - logit(joules)` under each intervention, not
  just top-1, because the model is likely confident on clean unit prompts.
- **Layer 32 SAE remains suspect.** Keep `--layers 4 8 12 16 20 24 28`.
- **Comparison across behaviours.** The value of this notebook for the report is the
  cross-behaviour comparison: does the units mechanism localize more cleanly to SAE
  features than carry did? Same tooling, so the contrast is apples-to-apples.